# Video Object Tracking on MOT16

**Group 4** | COMP-SCI 5567 | Faster R-CNN detection with Siamese re-identification

Runs top to bottom in about 25 minutes on a T4. Every figure is written to
`outputs/` at presentation resolution, ready to drop into the slides.

**Before running: Runtime -> Change runtime type -> T4 GPU.**


## 1. Setup


In [ ]:
import pathlib, sys, importlib, subprocess

REPO = "https://github.com/ben-blake/cs5567-project.git"
ROOT = pathlib.Path("/content/cs5567-project")

if not ROOT.exists():
    !git clone -q {REPO} {ROOT}
!git -C {ROOT} pull -q
!pip install -q -e {ROOT}

# A fresh editable install is often invisible to the running kernel, which
# shows up as "No module named mot". Adding src/ avoids a runtime restart.
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
importlib.invalidate_caches()
%cd {ROOT}

import mot
print("package:", mot.__file__)


In [ ]:
import time
import numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image

from mot.config import PATHS, get_device, describe_environment
from mot.data.mot16 import find_sequences, group_by_frame
from mot.data.datasets import ReIDPairDataset
from mot.data.augment import reid_transform
from mot.detect.faster_rcnn import build_pretrained_person_detector, detect, build_model, parameter_summary
from mot.reid.siamese import SiameseNetwork, CROP_SIZE, cosine_similarity_matrix, embed_crops
from mot.reid.train import train_reid, pair_accuracy
from mot.track.tracker import Tracker
from mot.eval.metrics import score_tracking, target_coverage
from mot.pipeline import TrackingPipeline, crop_people, render_results, results_to_mot_format
from torch.utils.data import DataLoader

PATHS.ensure()
DEVICE = get_device()
FIGDIR = PATHS.outputs
print(describe_environment())
print("figures ->", FIGDIR)

# House style, so every figure looks like it belongs to the same deck.
plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 11, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
})
UMKC_BLUE, ACCENT = "#005583", "#E08B2E"


## 2. Data

MOT16 is 1.9 GB and takes a few minutes. The Re-ID crops are cut from MOT16's
own ground truth, so they need no extra download and match the footage we track.


In [ ]:
!python scripts/download_data.py --mot16


In [ ]:
SEQUENCE_NAME = "MOT16-02"
MAX_FRAMES    = 600          # 20 seconds at 30 fps

sequences = find_sequences(PATHS.mot16 / "train")
sequence  = [s for s in sequences if s.name == SEQUENCE_NAME][0]
gt        = sequence.ground_truth()
by_frame  = group_by_frame(gt)

print(f"{len(sequences)} training sequences")
print(f"{sequence.name}: {sequence.info.seq_length} frames, "
      f"{sequence.info.im_width}x{sequence.info.im_height}, {sequence.info.frame_rate} fps")
print(f"ground truth: {len(gt)} pedestrian boxes across "
      f"{len({a.track_id for a in gt})} identities")

first = gt[0]
print(f"\nexample row -> frame {first.frame}, id {first.track_id}, "
      f"box {tuple(round(v,1) for v in first.box.as_xyxy())}, "
      f"visibility {first.visibility}")


### Figure 1: ground truth versus our detections


In [ ]:
from mot.viz.draw import draw_tracks

detector = build_pretrained_person_detector().eval().to(DEVICE)

FRAME = 1
image = Image.open(sequence.frame_path(FRAME)).convert("RGB")
anns  = by_frame[FRAME]

tensor = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0
det = detect(detector, [tensor], score_threshold=0.7, device=DEVICE)[0]

truth = draw_tracks(image,
                    np.array([a.box.as_xyxy() for a in anns], dtype=np.float32),
                    [a.track_id for a in anns])
ours  = draw_tracks(image, det["boxes"].numpy(),
                    list(range(1, len(det["boxes"]) + 1)),
                    det["scores"].tolist())

fig, axes = plt.subplots(2, 1, figsize=(11, 10))
for ax, img, title in ((axes[0], truth, f"Ground truth: {len(anns)} annotated people"),
                       (axes[1], ours,  f"Faster R-CNN, no fine-tuning: {len(det['boxes'])} detections")):
    ax.imshow(img); ax.axis("off"); ax.set_title(title, fontsize=12)
fig.tight_layout()
fig.savefig(FIGDIR / "fig1_detection.png")
plt.show()

print(f"annotated {len(anns)}, detected {len(det['boxes'])} on frame {FRAME}")


## 3. Re-identification

The Siamese network maps a person crop to a 128-dimensional unit vector, so
that crops of the same person land close together.


In [ ]:
summary = parameter_summary(build_model(pretrained=False, freeze_backbone=True))
print("Faster R-CNN few-shot budget")
for k, v in summary.items():
    print(f"  {k:10s} {v:>12,}")
print(f"  trainable share: {100*summary['trainable']/summary['total']:.1f}%")

reid_untrained = SiameseNetwork().eval()
print(f"\nSiamese network: {sum(p.numel() for p in reid_untrained.parameters()):,} "
      f"parameters, crops {CROP_SIZE[0]}x{CROP_SIZE[1]}")


In [ ]:
!python scripts/download_data.py --mot16-crops --max-sequences 3


In [ ]:
crop_root  = PATHS.data / "mot16_reid_crops"
identities = {}
for seq_dir in sorted(crop_root.iterdir()):
    if seq_dir.is_dir():
        for path in sorted(seq_dir.glob("*.jpg")):
            identities.setdefault(f"{seq_dir.name}_{path.name.split('_')[0]}", []).append(path)
identities = {i: p for i, (_, p) in enumerate(sorted(identities.items()))}
print(f"{len(identities)} identities, {sum(len(v) for v in identities.values())} crops")

train_loader = DataLoader(
    ReIDPairDataset(identities, transform=reid_transform(True), pairs_per_epoch=4000),
    batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(
    ReIDPairDataset(identities, transform=reid_transform(False), pairs_per_epoch=800, seed=7),
    batch_size=64, num_workers=2)

reid = SiameseNetwork()
acc_before = pair_accuracy(reid, val_loader, device=DEVICE)
print(f"pair accuracy before training: {acc_before:.3f}  (0.5 is chance)")


In [ ]:
history = train_reid(reid, train_loader, epochs=8, device=DEVICE,
                     val_loader=val_loader,
                     checkpoint_path=PATHS.checkpoints / "reid_siamese.pt",
                     log_every=0)
acc_after = history[-1]["val_accuracy"]
print(f"\npair accuracy {acc_before:.3f} -> {acc_after:.3f}")


### Figure 2: re-identification training


In [ ]:
epochs = [h["epoch"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(epochs, [h["loss"] for h in history], marker="o", color=UMKC_BLUE)
ax1.set_xlabel("epoch"); ax1.set_ylabel("contrastive loss"); ax1.set_title("Training loss")

ax2.plot(epochs, [h["val_accuracy"] for h in history], marker="o", color=UMKC_BLUE,
         label="validation")
ax2.axhline(0.5, ls="--", c="gray", label="chance")
ax2.set_xlabel("epoch"); ax2.set_ylabel("pair accuracy"); ax2.set_ylim(0.4, 1.0)
ax2.set_title(f"Held-out pairs: {acc_before:.3f} to {acc_after:.3f}"); ax2.legend()

fig.tight_layout()
fig.savefig(FIGDIR / "fig2_reid_training.png")
plt.show()


### Figure 3: what the embedding learned


In [ ]:
# Same crops, embedded before and after training. After training the diagonal
# (a crop against itself and its own identity) should stand out.
sample_ids = list(identities)[:8]
paths = [identities[i][0] for i in sample_ids] + [identities[i][1] for i in sample_ids]
transform = reid_transform(train=False)
batch = torch.stack([transform(Image.open(p).convert("RGB")) for p in paths])

def similarity(model):
    return cosine_similarity_matrix(embed_crops(model.eval(), batch, device=DEVICE),
                                    embed_crops(model.eval(), batch, device=DEVICE)).numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, matrix, title in ((axes[0], similarity(reid_untrained), "Before training"),
                          (axes[1], similarity(reid),           "After training")):
    im = ax.imshow(matrix, cmap="viridis", vmin=-1, vmax=1)
    ax.set_title(title); ax.set_xlabel("crop"); ax.set_ylabel("crop"); ax.grid(False)
    fig.colorbar(im, ax=ax, fraction=0.046, label="cosine similarity")
fig.suptitle("Two crops per person, 8 people. Blocks on the diagonal are the same person.",
             fontsize=10)
fig.tight_layout()
fig.savefig(FIGDIR / "fig3_reid_similarity.png")
plt.show()


## 4. Tracking

Detection, embedding and association over the whole sequence. The shipped
tracker uses overlap plus appearance, with re-acquisition disabled: see
`docs/RESULTS.md` for why.


In [ ]:
def run(reid_model, weight, label):
    pipeline = TrackingPipeline(detector, reid=reid_model,
                               tracker=Tracker(appearance_weight=weight),
                               score_threshold=0.7, device=DEVICE)
    started = time.time()
    results = pipeline.run_sequence(sequence, max_frames=MAX_FRAMES, progress=False)
    seconds = time.time() - started
    print(f"  {label}: {len(results)} frames in {seconds:.0f}s ({len(results)/seconds:.1f} FPS)")
    return results, seconds

print("tracking...")
results, elapsed = run(reid, 0.5, "with Re-ID  ")
results_no_reid, _ = run(None, 0.0, "overlap only")


In [ ]:
gt_eval = {f: (np.array([a.box.as_xyxy() for a in anns], dtype=np.float32),
               [a.track_id for a in anns])
           for f, anns in by_frame.items() if f <= MAX_FRAMES}
pred    = {r.frame: (r.boxes, r.track_ids) for r in results}
pred_no = {r.frame: (r.boxes, r.track_ids) for r in results_no_reid}

S, S_no = score_tracking(gt_eval, pred).as_dict(), score_tracking(gt_eval, pred_no).as_dict()
S["IDs"]    = len({i for r in results for i in r.track_ids})
S_no["IDs"] = len({i for r in results_no_reid for i in r.track_ids})

print(f"{'metric':10s} {'with Re-ID':>12s} {'overlap only':>14s}")
for k in ("MOTA", "MOTP", "TP", "FP", "FN", "IDSW", "IDs"):
    print(f"{k:10s} {S[k]:>12.3f} {S_no[k]:>14.3f}")
print(f"\nrecall {S['TP']/(S['TP']+S['FN']):.3f}   throughput {len(results)/elapsed:.1f} FPS")


## 5. How well is one person tracked?

The rubric asks for one target pedestrian tracked without being lost. That is a
claim about a real person, so it is measured against the ground truth rather
than against whichever predicted track happened to live longest.


In [ ]:
coverage = target_coverage(gt_eval, pred)
best     = coverage[0]
long_lived = [c for c in coverage if c.gt_frames >= 100]
median     = float(np.median([c.coverage for c in long_lived]))

print(f"best: ground-truth person {best.gt_id} held by predicted id {best.best_pred_id} "
      f"for {best.covered_frames}/{best.gt_frames} frames ({100*best.coverage:.1f}%)")
print(f"median across {len(long_lived)} people visible 100+ frames: {100*median:.1f}%")

TARGET_PRED = best.best_pred_id


### Figure 4: coverage across every person


In [ ]:
values = sorted((100*c.coverage for c in long_lived), reverse=True)

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(range(len(values)), values,
       color=[ACCENT if v == values[0] else UMKC_BLUE for v in values])
ax.axhline(100*median, ls="--", c="gray", label=f"median {100*median:.1f}%")
ax.set_xlabel(f"ground-truth person (n={len(values)}, visible 100+ frames)")
ax.set_ylabel("best single-ID coverage (%)")
ax.set_title(f"One person reaches {values[0]:.1f}%. Most do not, and that is the recall problem.")
ax.legend()
fig.tight_layout()
fig.savefig(FIGDIR / "fig4_coverage.png")
plt.show()


## 6. The demo video

A short clip over the stretch where the target is present, with the target in
colour and everyone else in grey.


In [ ]:
frames_on_target = [r.frame for r in results if TARGET_PRED in r.track_ids]
start, end = min(frames_on_target), max(frames_on_target)
print(f"predicted id {TARGET_PRED} spans frames {start} to {end}")

clip = [r for r in results if start <= r.frame <= min(end, start + 900)]
video = render_results(sequence, clip, FIGDIR / "demo_target_clip.mp4",
                       highlight_id=TARGET_PRED)
full  = render_results(sequence, results, FIGDIR / "demo_full.mp4",
                       highlight_id=TARGET_PRED)
(FIGDIR / "tracks.txt").write_text(results_to_mot_format(results))

print(f"clip: {video.name}  {len(clip)} frames, "
      f"{len(clip)/sequence.info.frame_rate:.0f}s, {video.stat().st_size/1024**2:.1f} MB")
print(f"full: {full.name}  {full.stat().st_size/1024**2:.1f} MB")


### Figure 5: the target across the clip


In [ ]:
from mot.viz.draw import draw_tracks as _draw

picks = [clip[i] for i in np.linspace(0, len(clip)-1, 4).astype(int)]
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, r in zip(axes.ravel(), picks):
    img = Image.open(sequence.frame_path(r.frame)).convert("RGB")
    ax.imshow(_draw(img, r.boxes, r.track_ids, highlight_id=TARGET_PRED))
    ax.axis("off"); ax.set_title(f"frame {r.frame}", fontsize=10)
fig.suptitle(f"Target id {TARGET_PRED} in colour, other tracks in grey", fontsize=12)
fig.tight_layout()
fig.savefig(FIGDIR / "fig5_target_strip.png")
plt.show()


## 7. Collect everything for the slides

Copy these into `slides/figures/` in the repo and the deck will pick them up
automatically.


In [ ]:
import shutil
from mot.config import drive_mounted

artifacts = sorted(p for p in FIGDIR.glob("*") if p.is_file())
print(f"{len(artifacts)} files in {FIGDIR}\n")
for p in artifacts:
    print(f"  {p.name:28s} {p.stat().st_size/1024**2:7.2f} MB")

if drive_mounted():
    dest = pathlib.Path("/content/drive/MyDrive/cs5567/presentation")
    dest.mkdir(parents=True, exist_ok=True)
    for p in artifacts:
        shutil.copy2(p, dest / p.name)
    print(f"\ncopied to {dest}")
else:
    print("\nDrive not mounted: download from the Files panel on the left.")

print("\nSlide figures expected at slides/figures/:")
for n in ("fig1_detection.png", "fig2_reid_training.png", "fig3_reid_similarity.png",
          "fig4_coverage.png", "fig5_target_strip.png"):
    print(f"  {n:28s} {'ready' if (FIGDIR/n).exists() else 'MISSING'}")
